# DCIR Setup Leveraging EDB Configuration Format

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import os
import tempfile
import time

from ansys.aedt.core import Hfss3dLayout
from ansys.aedt.core.examples.downloads import download_file
from pyedb import Edb


Define constants.

In [2]:
AEDT_VERSION = "2026.1"
NG_MODE = False

Download the example BGA Package design.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = download_file(source=r"pyaedt/edb/BGA_Package.aedb", local_path=temp_folder.name)

## Load example layout

In [4]:
edbapp = Edb(edbpath=file_edb, version=AEDT_VERSION)

PyEDB INFO: gRPC mode enabled.


PyEDB INFO: Logger is initialized in EDB.


PyEDB INFO: legacy v0.88.dev0


PyEDB INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


WARNING - The installed EDB_RPC_Server does not support shared-memory IPC. Falling back to standard gRPC communication. Update to the latest version of AEDT for improved performance.


PyEDB INFO: Grpc session started: pid=9176


... (3 outputs left out)

PyEDB INFO: Cell BGA_Package Opened


PyEDB INFO: Refreshing the Components dictionary.


PyEDB INFO: Builder was initialized.


PyEDB INFO: EDB initialized.


## Create config file

Define Component with solder balls.

In [5]:
cfg_components = [
    {
        "reference_designator": "FCHIP",
        "part_type": "other",
        "solder_ball_properties": {
            "shape": "cylinder",
            "diameter": "0.1mm",
            "height": "0.085mm",
            "chip_orientation": "chip_up",
        },
        "port_properties": {
            "reference_offset": 0,
            "reference_size_auto": False,
            "reference_size_x": 0,
            "reference_size_y": 0,
        },
    },
    {
        "reference_designator": "BGA",
        "part_type": "io",
        "solder_ball_properties": {
            "shape": "cylinder",
            "diameter": "0.45mm",
            "height": "0.45mm",
            "chip_orientation": "chip_down",
        },
        "port_properties": {
            "reference_offset": 0,
            "reference_size_auto": False,
            "reference_size_x": 0,
            "reference_size_y": 0,
        },
    },
]

Define Pin Groups on Components.

In [6]:
cfg_pin_groups = [
    {"name": "BGA_VSS", "reference_designator": "BGA", "net": "VSS"},
    {"name": "BGA_VDD", "reference_designator": "BGA", "net": "VDD"},
]

Define sources.

In [7]:
cfg_sources = [
    {
        "name": "FCHIP_Current",
        "reference_designator": "FCHIP",
        "type": "current",
        "magnitude": 2,
        "distributed": True,
        "positive_terminal": {"net": "VDD"},
        "negative_terminal": {"nearest_pin": {"reference_net": "VSS", "search_radius": 5e-3}},
    },
    {
        "name": "BGA_Voltage",
        "reference_designator": "BGA",
        "type": "voltage",
        "magnitude": 1,
        "positive_terminal": {"pin_group": "BGA_VDD"},
        "negative_terminal": {"pin_group": "BGA_VSS"},
    },
]

Define DCIR setup.

In [8]:
cfg_setups = [
    {
        "name": "siwave_dc",
        "type": "siwave_dc",
        "dc_slider_position": 1,
        "dc_ir_settings": {"export_dc_thermal_data": True},
    }
]

Create final configuration.

In [9]:
cfg = {
    "components": cfg_components,
    "sources": cfg_sources,
    "pin_groups": cfg_pin_groups,
    "setups": cfg_setups,
}

Create the config file.

In [10]:
file_json = os.path.join(temp_folder.name, "edb_configuration.json")
with open(file_json, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

## Apply Config file

Apply configuration to the example layout

In [11]:
edbapp.configuration.load(config_file=file_json)
edbapp.configuration.run()

PyEDB INFO: Updating nets finished. Time lapse 0:00:00


PyEDB INFO: Updating components finished. Time lapse 0:00:00.079772


C:\actions-runner\_work\pyaedt-examples\pyaedt-examples\.venv\lib\site-packages\pyedb\configuration\cfg_pin_groups.py:284: FutureWarning: Call to deprecated function create_pin_group. use edb.components.create_pin_group method instead
  if not self._pedb.siwave.create_pin_group(self.reference_designator, pins, self.name):


PyEDB INFO: Creating pin groups finished. Time lapse 0:00:00.555216


PyEDB INFO: Placing sources finished. Time lapse 0:00:13.689572


PyEDB INFO: Applying materials finished. Time lapse 0:00:00


... (12 outputs left out)

PyEDB INFO: Applying setups completed in 0.0159 seconds.


PyEDB INFO: Applying operations completed in 0.0000 seconds.


C:\actions-runner\_work\pyaedt-examples\pyaedt-examples\.venv\lib\site-packages\pyedb\configuration\configuration.py:324: FutureWarning: Call to deprecated function create_siwave_dc_setup. Use simulation_setups.create_siwave_dcir_setup() instead.
  edb_setup = self._pedb.create_siwave_dc_setup(


True

Save and close EDB.

In [12]:
edbapp.save()
edbapp.close()

PyEDB INFO: RPC session released (open databases: 0)


True

The configured EDB file is saved in a temp folder.

In [13]:
print(temp_folder.name)

C:\Users\ansys\AppData\Local\Temp\tmp4yd4oyj0.ansys


## Load edb into HFSS 3D Layout.

In [14]:
h3d = Hfss3dLayout(edbapp.edbpath, version=AEDT_VERSION, non_graphical=NG_MODE, new_desktop=True)

PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].


PyAEDT INFO: PyAEDT version 1.10.dev0.


PyAEDT INFO: Initializing new Desktop session.


PyAEDT INFO: AEDT version 2026.1.


PyAEDT INFO: New AEDT session is starting on gRPC port 63759.


PyAEDT INFO: Starting new AEDT gRPC session on port 63759.


... (4 outputs left out)

PyAEDT INFO: Non-graphical mode detected. Disabling Desktop logs.


PyAEDT INFO: EDB folder C:\Users\ansys\AppData\Local\Temp\tmp4yd4oyj0.ansys\pyaedt\edb\BGA_Package.aedb has been imported to project BGA_Package


PyAEDT INFO: Active Design set to 0;BGA_Package


PyAEDT INFO: AEDT objects correctly read


Solve.

In [15]:
h3d.analyze(setup="siwave_dc")

PyAEDT INFO: Project BGA_Package Saved correctly


PyAEDT INFO: Solving design setup siwave_dc


PyAEDT INFO: Design setup siwave_dc solved correctly in 0.0h 0.0m 55.0s


True

Plot insertion loss.

In [16]:
h3d.post.create_fieldplot_layers_nets(layers_nets=[["VDD_C1", "VDD"]], quantity="Voltage", setup="siwave_dc")

PyAEDT INFO: Parsing C:\Users\ansys\AppData\Local\Temp\tmp4yd4oyj0.ansys\pyaedt\edb\BGA_Package.aedt.


PyAEDT INFO: File C:\Users\ansys\AppData\Local\Temp\tmp4yd4oyj0.ansys\pyaedt\edb\BGA_Package.aedt correctly loaded. Elapsed time: 0m 0sec


PyAEDT INFO: aedt file load time 0.07276368141174316


PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec


PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec


PyAEDT INFO: Loading Modeler.


... (17 outputs left out)

PyEDB INFO: Builder was initialized.


PyEDB INFO: EDB initialized.


PyAEDT INFO: Active Design set to 0;BGA_Package


Class: ansys.aedt.core.visualization.post.field_data.FieldPlot

## Release AEDT

In [17]:
h3d.close_desktop()

PyAEDT INFO: Desktop has been released and closed.


True

In [18]:
# Wait 3 seconds to allow AEDT to shut down before cleaning the temporary directory.
time.sleep(3)

## Clean up

All project files are saved in the folder ``temp_folder.name``.
If you've run this example as a Jupyter notebook, you
can retrieve those project files. The following cell
removes all temporary files, including the project folder.

In [19]:
temp_folder.cleanup()